# Body-frame increment Flow Matching (`fm_rel_inc`)

Same initial-camera-relative conditioning, data, architecture, optimiser, steps, seed, solver and K. Only the generated quantity changes: absolute poses vs body-frame increments (`dR_i = R_iᵀR_{i+1}`, `dp_i = R_iᵀ(p_{i+1} − p_i)`), reconstructed by sequential composition from the anchored start. **No endpoint projection or correction.**

Question (H2): does increment generation change smoothness and/or multimodal fidelity, and what endpoint drift does it cost? A jerk reduction with worse endpoints is a trade-off, not automatically a failure.

Steps: (1) setup, (2) tests, (3) re-evaluate `fm_rel_pose` from Drive + solver diagnostics, (3b) retrain `fm_abs_pose` and build the fully matched gate-3 comparison, (4) train `fm_rel_inc`, (5) evaluate, (6) matched comparison, (7) manifest + pack. Everything is saved to Google Drive, so a disconnect only costs the current cell: re-run it and training resumes. Sprint default `DO_ABS_RETRAIN = False`: ≈ 35–45 min on a T4 (two training runs). With it on: +~30 min.

## 1. Code + Drive

In [ ]:
import os, sys, glob, json, subprocess
REPO_URL = ""  # Optional Git remote URL for Colab; otherwise upload the repository ZIP.
IN_COLAB = "google.colab" in sys.modules

if not os.path.isdir("camera_motion"):
    candidates = sorted(os.path.dirname(p) for p in glob.glob("*/camera_motion") if os.path.isdir(p))
    if not candidates and IN_COLAB:
        if REPO_URL:
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "camera-motion-flow-matching"], check=True)
        else:
            from google.colab import files
            from zipfile import ZipFile
            uploaded = files.upload()
            if not uploaded:
                raise RuntimeError("Upload the repository ZIP or specify REPO_URL.")
            with ZipFile(next(iter(uploaded))) as zf:
                zf.extractall(".")
        candidates = sorted(os.path.dirname(p) for p in glob.glob("*/camera_motion") if os.path.isdir(p))
    if candidates:
        os.chdir(candidates[0])
assert os.path.isdir("camera_motion"), f"Cannot find repository root: {os.getcwd()}"
sys.path.insert(0, os.getcwd())
print("repository root:", os.getcwd())


RUNS = "runs"
if IN_COLAB:                                       # default: everything goes to Drive
    from google.colab import drive
    drive.mount("/content/drive")
    RUNS = "/content/drive/MyDrive/camera_motion_runs"
os.makedirs(RUNS, exist_ok=True)
SEED = 0
POSE = {f: f"{RUNS}/fm_rel_pose_{f}_s{SEED}" for f in ("E1a", "E1b")}
INC  = {f: f"{RUNS}/fm_rel_inc_{f}_s{SEED}" for f in ("E1a", "E1b")}
ABS  = {f: f"{RUNS}/fm_abs_pose_{f}_s{SEED}" for f in ("E1a", "E1b")}
DO_ABS_RETRAIN = False   # sprint mode: off. True = retrain abs_pose for a fully matched m2 comparison (+~30 min)
for f, d in POSE.items():
    print(f, "rel_pose checkpoint:", "OK" if os.path.exists(f"{d}/ckpt_last.pt") else "MISSING -> " + d)
import torch
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 2. Tests

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider"], capture_output=True, text=True)
print(r.stdout[-600:]); assert r.returncode == 0, r.stderr[-1500:]

## 3. `fm_rel_pose`: re-evaluate (adds new report fields; numbers are deterministic) + solver diagnostics
The solver sweep uses the same checkpoint, conditions, K and seed for every setting. If jerk/GT falls a lot with NFE, the excess jerk is integration error.

In [ ]:
!python -u scripts/eval_fm.py --run {POSE['E1a']} --K 200 --nfe 50
!python -u scripts/eval_fm.py --run {POSE['E1b']} --K 200 --nfe 50
!python -u scripts/diagnose_solver.py --run {POSE['E1a']} --panel eval_id --grid euler:20 euler:50 euler:100 midpoint:50 midpoint:100

## 3b. Gate-3 matched comparison: retrain `fm_abs_pose` (same seed/budget), evaluate with m2
The original checkpoints were lost, so this is a retrain with the identical command. GPU kernels are not bit-deterministic, so expect small differences from the gate-2 numbers; the m2 comparison below is the one to report.

In [ ]:
if DO_ABS_RETRAIN:
    for f in ("E1a", "E1b"):
        !python -u scripts/train_fm.py --rep abs_pose --family {f} --n-train 10000 --steps 20000 --batch 256 --seed {SEED} --log-every 2000 --out {ABS[f]}
        !python -u scripts/eval_fm.py --run {ABS[f]} --K 200 --nfe 50

In [ ]:
if DO_ABS_RETRAIN:
    EV = "eval_m2_K200_nfe50_euler"
    CMP3 = f"{RUNS}/comparisons/gate3_abs_vs_rel_matched"
    for f in ("E1a", "E1b"):
        !python -u scripts/matched_comparison.py --family {f} --a {ABS[f]}/{EV} --b {POSE[f]}/{EV} --out {CMP3} --label-a abs_pose --label-b rel_pose
        !python -u scripts/compare_runs.py --a {ABS[f]}/{EV} --b {POSE[f]}/{EV}

## 4. Train `fm_rel_inc` (identical budget). Re-running resumes from Drive.

In [ ]:
!python -u scripts/train_fm.py --rep rel_inc --family E1a --n-train 10000 --steps 20000 --batch 256 --seed {SEED} --log-every 2000 --out {INC['E1a']}

In [ ]:
!python -u scripts/train_fm.py --rep rel_inc --family E1b --n-train 10000 --steps 20000 --batch 256 --seed {SEED} --log-every 2000 --out {INC['E1b']}

## 5. Evaluate `fm_rel_inc` (+ the same solver sweep, for a like-for-like jerk comparison)

In [ ]:
!python -u scripts/eval_fm.py --run {INC['E1a']} --K 200 --nfe 50
!python -u scripts/eval_fm.py --run {INC['E1b']} --K 200 --nfe 50
!python -u scripts/diagnose_solver.py --run {INC['E1a']} --panel eval_id --grid euler:20 euler:50 euler:100 midpoint:50 midpoint:100

## 6. Matched comparison: a = `fm_rel_pose`, b = `fm_rel_inc`
Paired 95% CIs over held-out conditions. OOD-position is reported but is NOT discriminative here: both models use the relative frame.

In [ ]:
EV = "eval_m2_K200_nfe50_euler"
CMP = f"{RUNS}/comparisons/gate4_rel_pose_vs_rel_inc"
for f in ("E1a", "E1b"):
    !python -u scripts/matched_comparison.py --family {f} --a {POSE[f]}/{EV} --b {INC[f]}/{EV} --out {CMP} --label-a rel_pose --label-b rel_inc
    !python -u scripts/compare_runs.py --a {POSE[f]}/{EV} --b {INC[f]}/{EV}

In [ ]:
from IPython.display import Image, display, Markdown
for f in ("E1a", "E1b"):
    display(Markdown(open(f"{CMP}/{f}_matched_comparison.md").read()))
    if DO_ABS_RETRAIN:
        display(Markdown(open(f"{RUNS}/comparisons/gate3_abs_vs_rel_matched/{f}_matched_comparison.md").read()))
for p in sorted(glob.glob(f"{CMP}/*.png")) + sorted(glob.glob(f"{POSE['E1a']}/diagnostics/*.png")) + sorted(glob.glob(f"{INC['E1a']}/diagnostics/*.png")):
    print(p); display(Image(filename=p))
for d in (INC["E1a"], INC["E1b"]):
    for p in sorted(glob.glob(f"{d}/{EV}/figures/*.png")):
        print(p); display(Image(filename=p))

## 6b. Demo videos + benchmark (Deliverables 1 & 2)
Rendered on CPU from the samples saved by the evaluations.

In [ ]:
DEMO, BENCH = f"{RUNS}/demo", f"{RUNS}/benchmark"
for name, d in (("rel_pose", POSE), ("rel_inc", INC)):
    !python -u scripts/render_demo.py modes --eval-dir {d['E1a']}/{EV} --family E1a --out {DEMO}
    !python -u scripts/render_demo.py gt_vs_gen --eval-dir {d['E1b']}/{EV} --family E1b --out {DEMO}
!python -u scripts/make_benchmark.py --run rel_pose={POSE['E1a']}/{EV} --run rel_pose={POSE['E1b']}/{EV} --run rel_inc={INC['E1a']}/{EV} --run rel_inc={INC['E1b']}/{EV} --out {BENCH} --condition 1
from IPython.display import Video
display(Markdown(open(f"{BENCH}/benchmark_table.md").read()))
for p in sorted(glob.glob(f"{BENCH}/*.png")):
    display(Image(filename=p))
for v in sorted(glob.glob(f"{DEMO}/*.mp4")):
    print(v); display(Video(v, embed=True, width=900))

## 7. Manifest + pack (no checkpoints)

In [ ]:
!python -u scripts/write_manifest.py --runs {RUNS} --out {RUNS}/experiment_manifest.json
import zipfile
with zipfile.ZipFile("gate4_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    z.write(f"{RUNS}/experiment_manifest.json", "experiment_manifest.json")
    extra = (list(ABS.values()) + [f"{RUNS}/comparisons/gate3_abs_vs_rel_matched"]) if DO_ABS_RETRAIN else []
    for d in list(POSE.values()) + list(INC.values()) + [CMP, DEMO, BENCH] + extra:
        for f in glob.glob(f"{d}/**/*", recursive=True):
            if os.path.isdir(f) or f.endswith(".pt"):
                continue
            z.write(f, os.path.relpath(f, RUNS))
print("gate4_results.zip", round(os.path.getsize("gate4_results.zip") / 2**20, 1), "MB")
if IN_COLAB:
    from google.colab import files; files.download("gate4_results.zip")